In [1]:
# Imports and basic setup
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use("fivethirtyeight")

from tqdm import tqdm
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchmetrics import ROC
from torchmetrics.classification import BinaryAccuracy




In [2]:
# Helper class for data loading
class Datapreparation:
    def __init__(self, root_path):
        self.root_path = root_path

    def get_dataframe(self, filename):
        return pd.read_csv(os.path.join(self.root_path, filename))

    def summary(self, text, df):
        summary = pd.DataFrame(df.dtypes, columns=["dtypes"])
        summary["null"] = df.isnull().sum()
        summary["unique"] = df.nunique()
        summary["min"] = df.min()
        summary["median"] = df.median()
        summary["max"] = df.max()
        summary["mean"] = df.mean()
        summary["std"] = df.std()
        summary["duplicate"] = df.duplicated().sum()
        return summary

    def random_split_data(self, X, y):
        return train_test_split(X, y, test_size=0.20, random_state=42)

    def standardization_data(self, X_data):
        scaler = StandardScaler()
        return scaler.fit_transform(X_data)


data = Datapreparation("/kaggle/input/playground-series-s3e18")
train = data.get_dataframe("train.csv")




In [3]:
# Quick overview (optional)
_ = data.summary("train", train)




In [4]:
# Prepare features and targets
y1 = train["EC1"]
y2 = train["EC2"]
train.drop(
    columns=["id", "EC1", "EC2", "EC3", "EC4", "EC5", "EC6"], axis=1, inplace=True
)
X = train.copy()

X_train, X_val, y1_train, y1_val = data.random_split_data(X, y1)
_, _, y2_train, y2_val = data.random_split_data(X, y2)  # same splits for EC2

std_X_train = data.standardization_data(X_train)
std_X_val = data.standardization_data(X_val)




In [5]:
# Tensor conversion utilities
class Tensoroperations:
    def convert_to_tensor(self, X, y=None):
        X_tensor = torch.from_numpy(X).float()
        if y is not None:
            y_tensor = torch.from_numpy(y).float()
            return X_tensor, y_tensor
        return X_tensor

    def convert_to_test_tensor(self, X):
        return torch.from_numpy(X).float()

    def get_dataloaders(self, train_dataset, val_dataset):
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)
        return train_loader, val_loader


tenops = Tensoroperations()




In [6]:
# Custom Dataset classes
class CustomDataset(Dataset):
    def __init__(self, X_data, y_data=None):
        self.X_data = X_data
        self.y_data = y_data

    def __getitem__(self, idx):
        if self.y_data is None:
            return self.X_data[idx]
        return self.X_data[idx], self.y_data[idx]

    def __len__(self):
        return len(self.X_data)




In [7]:
# Create tensors and datasets for both targets
X_tensor_train, y1_tensor_train = tenops.convert_to_tensor(std_X_train, y1_train.values)
X_tensor_val, y1_tensor_val = tenops.convert_to_tensor(std_X_val, y1_val.values)

train1_dataset = CustomDataset(X_tensor_train, y1_tensor_train)
val1_dataset = CustomDataset(X_tensor_val, y1_tensor_val)

X_tensor_train2, y2_tensor_train = tenops.convert_to_tensor(
    std_X_train, y2_train.values
)
X_tensor_val2, y2_tensor_val = tenops.convert_to_tensor(std_X_val, y2_val.values)

train2_dataset = CustomDataset(X_tensor_train2, y2_tensor_train)
val2_dataset = CustomDataset(X_tensor_val2, y2_tensor_val)




In [8]:
# Containers for ROC tracking (optional)
false_pos, true_pos = [], []




In [9]:
# Base model with shared utilities
class EnzymeClassificationBase(nn.Module):
    def _accuracy(self, outputs, labels):
        metric = BinaryAccuracy().to(device)
        return metric(outputs, labels.unsqueeze(1))

    def training_step(self, batch):
        features, labels = batch
        out = self(features)
        loss = F.binary_cross_entropy(out, labels.unsqueeze(1))
        return loss

    def validation_step(self, batch):
        features, labels = batch
        features = features.to(device)
        labels = labels.to(device)

        out = self(features)
        loss = F.binary_cross_entropy(out, labels.unsqueeze(1))
        acc = self._accuracy(out, labels)
        self._get_roc(out, labels)
        return {"Validation_loss": loss.detach(), "Validation_acc": acc}

    def _get_roc(self, output, labels):
        roc = ROC(task="binary").to(device)
        fpr, tpr, _ = roc(output, labels.long().unsqueeze(1))
        false_pos.append(fpr)
        true_pos.append(tpr)

    def validation_epoch_end(self, outputs):
        batch_losses = [x["Validation_loss"] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()
        batch_accs = [x["Validation_acc"] for x in outputs]
        epoch_acc = torch.stack(batch_accs).mean()
        return {
            "Validation_loss": epoch_loss.item(),
            "Validation_acc": epoch_acc.item(),
        }

    def epoch_end(self, epoch, result):
        if epoch % 10 == 0:
            print(
                f"Epoch [{epoch}], Train_loss: {result['Train_loss']:.4f}, "
                f"Validation_loss: {result['Validation_loss']:.4f}, "
                f"Validation_acc: {result['Validation_acc']:.4f}"
            )




In [10]:
# Define the concrete model architecture
n_input_dim = X_train.shape[1]
n_output = 1


class MultiClassificationNN(EnzymeClassificationBase):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_input_dim, 32),
            nn.LeakyReLU(),
            nn.Linear(32, 16),
            nn.LeakyReLU(),
            nn.Linear(16, 8),
            nn.LeakyReLU(),
            nn.Linear(8, n_output),
            nn.Sigmoid(),
        )

    def forward(self, xb):
        return self.network(xb)


model_EC1 = MultiClassificationNN()
model_EC2 = MultiClassificationNN()




In [11]:
# Device configuration and seeding
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

model_EC1 = model_EC1.to(device)
model_EC2 = model_EC2.to(device)




In [12]:
# Trainer utility
class Trainer:
    @torch.no_grad()
    def _evaluate(self, model, val_loader):
        model.eval()
        outputs = [model.validation_step(batch) for batch in val_loader]
        return model.validation_epoch_end(outputs)

    def fit(self, epochs, lr, model, train_loader, val_loader, opt_func):
        history = []
        optimizer = opt_func(model.parameters(), lr)
        for epoch in tqdm(range(epochs), desc="Training"):
            model.train()
            train_losses = []
            for batch in train_loader:
                features, labels = batch
                features = features.to(device)
                labels = labels.to(device)
                loss = model.training_step((features, labels))
                train_losses.append(loss)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            result = self._evaluate(model, val_loader)
            result["Train_loss"] = torch.stack(train_losses).mean().item()
            model.epoch_end(epoch, result)
            history.append(result)
        return history




In [13]:
# Train both models
num_epochs = 30
lr = 1e-4
opt_func = torch.optim.Adam

train_loader_ec1, val_loader_ec1 = tenops.get_dataloaders(train1_dataset, val1_dataset)
train_loader_ec2, val_loader_ec2 = tenops.get_dataloaders(train2_dataset, val2_dataset)

trainer = Trainer()
history_EC1 = trainer.fit(
    num_epochs, lr, model_EC1, train_loader_ec1, val_loader_ec1, opt_func
)
history_EC2 = trainer.fit(
    num_epochs, lr, model_EC2, train_loader_ec2, val_loader_ec2, opt_func
)




Training:   3%|▎         | 1/30 [00:00<00:24,  1.20it/s]

Epoch [0], Train_loss: 0.6621, Validation_loss: 0.6399, Validation_acc: 0.6656


Training:  37%|███▋      | 11/30 [00:06<00:10,  1.76it/s]

Epoch [10], Train_loss: 0.5815, Validation_loss: 0.5826, Validation_acc: 0.7002


Training:  70%|███████   | 21/30 [00:12<00:05,  1.76it/s]

...

Training:  37%|███▋      | 11/30 [00:06<00:11,  1.71it/s]

Epoch [10], Train_loss: 0.4979, Validation_loss: 0.4880, Validation_acc: 0.8064


Training:  70%|███████   | 21/30 [00:12<00:05,  1.76it/s]

Epoch [20], Train_loss: 0.4950, Validation_loss: 0.4869, Validation_acc: 0.8064


Training: 100%|██████████| 30/30 [00:17<00:00,  1.73it/s]


In [14]:
# Prepare test data
test = data.get_dataframe("test.csv")
test_ids = test["id"].values
test_features = test.drop(columns=["id"])

std_X_test = data.standardization_data(test_features)
X_tensor_test = tenops.convert_to_test_tensor(std_X_test)


class CustomDataTest(Dataset):
    def __init__(self, X_data):
        self.X_data = X_data

    def __getitem__(self, idx):
        return self.X_data[idx]

    def __len__(self):
        return len(self.X_data)


test_dataset = CustomDataTest(X_tensor_test)
test_dataloader = DataLoader(test_dataset, batch_size=64)




In [15]:
# Evaluation on test set
class Evaluate:
    def eval_test_data(self, model, test_loader):
        model.eval()
        preds = []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch)
                prob = torch.sigmoid(
                    out
                )  # model already applies sigmoid; this is harmless
                preds.append(prob.cpu())
        # flatten list of tensors to plain Python floats
        flat = [p.squeeze().item() for tensor in preds for p in tensor]
        return flat


evaluator = Evaluate()
ec1_preds = evaluator.eval_test_data(model_EC1, test_dataloader)
ec2_preds = evaluator.eval_test_data(model_EC2, test_dataloader)




In [16]:
# Create submission file
submission = pd.DataFrame({"id": test_ids, "EC1": ec1_preds, "EC2": ec2_preds})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
